<p align="center"><a href="https://www.plus2net.com/python/pandas-read_pickle.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas read_pickle(): Restore and Validate Trusted Pickle Files

Run cells in order. All sample data is embedded and synthetic. Every pickle read here was created within the notebook. Only load pickle files from trusted sources: unpickling can execute code. Temporary directories are cleaned automatically. Try the exercise before its solution.

## Read the original marks DataFrame

read_pickle() returns the saved object. Here it is a DataFrame with Ravi, Raju and Alex records; the notebook creates the trusted input itself. No download or uploaded pickle is required. See <a href="pandas-to_pickle.php">to_pickle()</a>.

In [ ]:
import pandas as pd
import pickle
import tempfile
import sqlite3
from pathlib import Path
from io import BytesIO, StringIO
df = pd.DataFrame({"NAME": ["Ravi", "Raju", "Alex"], "ID": [1, 2, 3],
    "MATH": [30, 40, 50], "ENGLISH": [20, 30, 40]})
print(df)
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "my_data.pkl"
    df.to_pickle(target)
    my_data = pd.read_pickle(target)
    pd.testing.assert_frame_equal(my_data, df)
    print("Restored rows:", len(my_data))

**Expected output**

```text
   NAME  ID  MATH  ENGLISH
0  Ravi   1    30       20
1  Raju   2    40       30
2  Alex   3    50       40
Restored rows: 3
```

## Read from a chosen folder

Resolve paths relative to the current working directory, or use an absolute path. A Windows raw path such as <code>Path(r"D:\my_data") / "my_data.pkl"</code> preserves backslashes. Check that the path points to a trusted file before unpickling.

In [ ]:
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "archive" / "my_data.pkl"
    target.parent.mkdir()
    df.to_pickle(target)
    restored = pd.read_pickle(target)
    pd.testing.assert_frame_equal(restored, df)
    print("Read archive/my_data.pkl")

**Expected output**

```text
Read archive/my_data.pkl
```

## Read inferred gzip compression

compression="infer" inspects recognized path extensions. Read the .pkl.gz file using the default. Do not guess compression from the .pkl suffix alone.

In [ ]:
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "my_data.pkl.gz"
    df.to_pickle(target)
    restored = pd.read_pickle(target)
    pd.testing.assert_frame_equal(restored, df)
    print("Gzip restored rows:", len(restored))

**Expected output**

```text
Gzip restored rows: 3
```

## Read explicit ZIP compression

Retain the original compression="zip" example. When ZIP bytes use a .pkl name, specify ZIP explicitly at read time. A ZIP or TAR archive must contain only one data file.

In [ ]:
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "my_data.pkl"
    df.to_pickle(target, compression="zip")
    restored = pd.read_pickle(target, compression="zip")
    pd.testing.assert_frame_equal(restored, df)
    print("Explicit ZIP restored rows:", len(restored))

**Expected output**

```text
Explicit ZIP restored rows: 3
```

## Read a binary buffer

Rewind a buffer before reading. read_pickle() reads binary pickle data, rather than a text StringIO buffer.

In [ ]:
buffer = BytesIO()
df.to_pickle(buffer)
buffer.seek(0)
restored = pd.read_pickle(buffer, compression=None)
pd.testing.assert_frame_equal(restored, df)
print("Buffer restored rows:", len(restored))

**Expected output**

```text
Buffer restored rows: 3
```

## Check which object was stored

A pickle need not contain a DataFrame. read_pickle() can return a Series or another Python object. After loading a trusted file, check the type before applying DataFrame methods. A post-load type check is a data contract, not a security check.

In [ ]:
series = pd.Series([30, 40, 50], index=[1, 2, 3], name="MATH")
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "marks.pkl"
    series.to_pickle(target)
    restored = pd.read_pickle(target)
    assert isinstance(restored, pd.Series)
    pd.testing.assert_series_equal(restored, series)
    print("Stored type:", type(restored).__name__)
    print(restored)

**Expected output**

```text
Stored type: Series
1    30
2    40
3    50
Name: MATH, dtype: int64
```

## Verify index and missing-value dtypes

Read_pickle restores saved dtypes and labels in this same-environment example. Check the artifact against the expected schema rather than inferring success from a filename.

In [ ]:
typed = pd.DataFrame({"units": pd.array([2, None], dtype="Int64"),
    "state": pd.Categorical(["ready", "review"])},
    index=pd.Index(["001", "002"], name="order_id"))
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "orders.pkl"
    typed.to_pickle(target)
    restored = pd.read_pickle(target)
    pd.testing.assert_frame_equal(restored, typed)
    print(restored.dtypes)
    assert restored.index.tolist() == ["001", "002"]
    assert restored["units"].isna().sum() == 1

**Expected output**

```text
units       Int64
state    category
dtype: object
```

## Validate a restored sales checkpoint

This synthetic sales checkpoint contains a pending amount. Preserve the raw records and separate accepted and review rows. The pickle is created within this example and is trusted. Continue with <a href="pandas-data-cleaning.php">Data Cleaning</a>.

In [ ]:
sales = pd.DataFrame({"id": ["001", "002", "003"], "amount": ["5.25", "20", "pending"]})
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "sales.pkl"
    sales.to_pickle(target)
    raw = pd.read_pickle(target)
assert isinstance(raw, pd.DataFrame) and raw.columns.tolist() == ["id", "amount"]
clean = raw.copy()
clean["amount"] = pd.to_numeric(raw["amount"], errors="coerce")
valid = clean["amount"].ge(0).fillna(False)
accepted = clean.loc[valid].copy()
review = raw.loc[~valid].copy()
print("Accepted total:", accepted["amount"].sum())
print("Review IDs:", review["id"].tolist())
assert accepted["amount"].sum() == 25.25
assert len(accepted) + len(review) == len(raw)

**Expected output**

```text
Accepted total: 25.25
Review IDs: ['003']
```

## Parameters, trusted files and compatibility

<div class="table-responsive"><table class="table table-bordered"><tr><th>Parameter</th><th>Purpose</th></tr><tr><td><code>filepath_or_buffer</code></td><td>Required path or binary readable buffer.</td></tr><tr><td><code>compression</code></td><td>infer by default; specify the format when the suffix does not identify it.</td></tr><tr><td><code>storage_options</code></td><td>Forwarded options for supported storage backends.</td></tr></table></div><p><strong>Trusted sources only:</strong> loading a pickle can execute code. Do not load an unknown attachment merely to inspect it. <strong>Missing file:</strong> check the working directory and source path. <strong>Wrong compression:</strong> match the writer settings. <strong>Compatibility errors:</strong> recreate the documented Python/Pandas environment or regenerate an interchange file in the producing environment. Pickle is not a permanent cross-version guarantee. <strong>Large files:</strong> read_pickle() loads an object and has no chunksize parameter. Consider chunked <a href="pandas-read_csv.php">CSV</a> or <a href="pandas-read_sql.php">SQL imports</a> for large workflows. See <a href="pickle.php">Python pickle</a> and <a href="pickle-mysql.php">database pickling</a>.</p>

## Exercise: restore and filter marks

Create a trusted pickle from df, restore it, then select NAME and MATH where MATH is at least 40. Verify the names Raju and Alex.

## Exercise solution

Read first, verify the restored object, then filter the required fields.

In [ ]:
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "exercise.pkl"
    df.to_pickle(target)
    restored = pd.read_pickle(target)
assert isinstance(restored, pd.DataFrame)
answer = restored.loc[restored["MATH"].ge(40), ["NAME", "MATH"]]
print(answer)
assert answer["NAME"].tolist() == ["Raju", "Alex"]

**Expected output**

```text
   NAME  MATH
1  Raju    40
2  Alex    50
```